# 03. 데이터 증강 — albumentations로 일반화 성능 높이기

**데이터 증강(Data Augmentation)** 은 원본 이미지에 뒤집기·회전·밝기 변화 등을 무작위로 적용해
학습 데이터를 인위적으로 다양하게 만드는 기법입니다. 모델이 다양한 변형을 보며 학습하므로
**과적합이 줄고 일반화 성능(처음 보는 데이터에 대한 정확도)이 향상**됩니다.

이 노트북에서는 `albumentations` 라이브러리로 증강을 적용하고,
**증강 없이 학습한 모델 vs 증강을 적용한 모델**의 테스트 정확도를 직접 비교합니다.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)

## 1. 증강 기법 정의

학습용 변환에는 무작위 증강을, 테스트용 변환에는 정규화만 적용합니다 (평가 시에는 변형하지 않음).

- `HorizontalFlip` : 좌우 반전
- `Affine` : 이동·확대축소·회전
- `RandomBrightnessContrast` : 밝기·대비 변화
- `Normalize` + `ToTensorV2` : 정규화 후 텐서 변환

In [ ]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(translate_percent=0.1, scale=(0.9, 1.1), rotate=(-15, 15), p=0.7),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

plain_tf = A.Compose([
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])
print('증강 정의 완료')

### 증강 결과 살펴보기

같은 이미지에 증강을 여러 번 적용하면 매번 다른 변형이 나옵니다. 직접 확인해 봅니다.

In [ ]:
raw_set = torchvision.datasets.CIFAR10(root='/workspace/data', train=True, download=True)
img, label = raw_set[7]
img_np = np.array(img)

fig, axes = plt.subplots(1, 6, figsize=(13, 2.5))
axes[0].imshow(img_np); axes[0].set_title('원본'); axes[0].axis('off')
for i in range(1, 6):
    aug = train_aug(image=img_np)['image']  # 텐서(C,H,W), 정규화됨
    show = aug * torch.tensor(std).view(3,1,1) + torch.tensor(mean).view(3,1,1)
    axes[i].imshow(show.permute(1,2,0).numpy().clip(0,1))
    axes[i].set_title(f'증강 {i}'); axes[i].axis('off')
plt.suptitle('동일 이미지에 무작위 증강 적용', fontsize=13)
plt.tight_layout(); plt.show()

## 2. 증강 적용 데이터셋

`albumentations`는 NumPy 이미지를 입력으로 받으므로, CIFAR-10(PIL 이미지)을 감싸
증강을 적용하는 데이터셋을 만듭니다.

In [ ]:
class AlbumentationsCIFAR10(torch.utils.data.Dataset):
    def __init__(self, base, transform):
        self.base = base          # transform 없는 CIFAR10
        self.transform = transform
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        img, label = self.base[idx]
        img = np.array(img)
        img = self.transform(image=img)['image']
        return img, label

base_train = torchvision.datasets.CIFAR10(root='/workspace/data', train=True, download=True)
base_test = torchvision.datasets.CIFAR10(root='/workspace/data', train=False, download=True)
print('데이터셋 준비 완료')

## 3. 증강 없이 vs 증강 적용 — 학습 비교

공정한 비교를 위해 **동일한 CNN 구조**를 두 번 학습합니다.
한 번은 증강 없이, 한 번은 증강을 적용해서. 그리고 같은 테스트셋으로 정확도를 비교합니다.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.5),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


def train_and_eval(train_tf, epochs=10):
    train_loader = torch.utils.data.DataLoader(
        AlbumentationsCIFAR10(base_train, train_tf), batch_size=128, shuffle=True, num_workers=2)
    test_loader = torch.utils.data.DataLoader(
        AlbumentationsCIFAR10(base_test, plain_tf), batch_size=256, shuffle=False, num_workers=2)

    model = SimpleCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

In [ ]:
print('증강 없이 학습 중...')
acc_plain = train_and_eval(plain_tf, epochs=10)
print(f'증강 없음 정확도: {acc_plain:.2f}%')

In [ ]:
print('증강 적용 학습 중...')
acc_aug = train_and_eval(train_aug, epochs=10)
print(f'증강 적용 정확도: {acc_aug:.2f}%')

In [ ]:
plt.figure(figsize=(5, 4))
plt.bar(['증강 없음', '증강 적용'], [acc_plain, acc_aug], color=['#bbbbbb', '#4c72b0'])
plt.ylabel('테스트 정확도 (%)')
plt.title('데이터 증강 효과 비교')
for i, v in enumerate([acc_plain, acc_aug]):
    plt.text(i, v + 0.3, f'{v:.1f}%', ha='center')
plt.ylim(0, 100)
plt.tight_layout(); plt.show()

### 정리

- 동일한 CNN을 **증강 없이 / 증강 적용** 두 조건으로 학습해 정확도를 비교했습니다.
- 일반적으로 증강을 적용한 쪽이 **일반화 성능이 더 높습니다** (과적합이 줄어들기 때문).
- 데이터 증강은 추가 데이터 수집 없이 성능을 끌어올리는 가장 손쉬운 방법 중 하나이며,
  특히 데이터가 적을수록 효과가 큽니다.
- `albumentations`는 빠르고 다양한 증강을 제공해 컴퓨터비전 실무에서 널리 쓰입니다.